# Baseline Metrics Summary and Best Model Selection

Read the four baseline `metrics.json` files, aggregate them automatically, and output the ranking and best model.

## 1. Configuration and Dependencies

In [ ]:
import json
from pathlib import Path
from typing import Any

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent.parent
BASE_DIR = PROJECT_ROOT / 'outputs' / 'baseline'
OUT_DIR = BASE_DIR / 'summary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_KEYS = ['resnet18', 'vgg16_bn', 'vit_b_16', 'swin_t']
PRIMARY_METRIC = 'test_macro_f1'  # can set to test_acc\n

print('[INFO] PROJECT_ROOT:', PROJECT_ROOT)
print('[INFO] BASE_DIR    :', BASE_DIR)
print('[INFO] OUT_DIR     :', OUT_DIR)

## 2. Utility Functions

In [ ]:
def safe_get(d: dict[str, Any], path: list[str], default: Any = None) -> Any:
    cur: Any = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def load_metrics(base_dir: Path, model_key: str) -> dict[str, Any]:
    metrics_path = base_dir / model_key / 'metrics.json'
    if not metrics_path.exists():
        return {
            'model_key': model_key,
            'exists': False,
            'test_acc': None,
            'test_macro_f1': None,
            'best_val_macro_f1': None,
            'metrics_path': str(metrics_path),
        }

    data = json.loads(metrics_path.read_text(encoding='utf-8'))
    return {
        'model_key': model_key,
        'exists': True,
        'test_acc': safe_get(data, ['test', 'acc']),
        'test_macro_f1': safe_get(data, ['test', 'macro_f1']),
        'best_val_macro_f1': safe_get(data, ['best_val_macro_f1']),
        'metrics_path': str(metrics_path),
    }

## 3. Aggregation, Ranking, and Best-Model Selection

In [ ]:
rows = [load_metrics(BASE_DIR, k) for k in MODEL_KEYS]
df = pd.DataFrame(rows)
display(df)

valid = df[df['exists'] & df['test_acc'].notna() & df['test_macro_f1'].notna()].copy()
if len(valid) == 0:
    print('[WARN] No comparable baseline metrics found. Run all four baseline notebooks first.')
else:
    secondary = 'test_acc' if PRIMARY_METRIC == 'test_macro_f1' else 'test_macro_f1'
    ranking = valid.sort_values(by=[PRIMARY_METRIC, secondary], ascending=False).reset_index(drop=True)
    best = ranking.iloc[0].to_dict()

    all_csv = OUT_DIR / 'baseline_metrics_all.csv'
    rank_csv = OUT_DIR / 'baseline_ranking.csv'
    best_json = OUT_DIR / 'best_baseline.json'

    df.to_csv(all_csv, index=False, encoding='utf-8')
    ranking.to_csv(rank_csv, index=False, encoding='utf-8')
    best_json.write_text(json.dumps({
        'primary_metric': PRIMARY_METRIC,
        'best_model_key': best['model_key'],
        'best_test_acc': best['test_acc'],
        'best_test_macro_f1': best['test_macro_f1'],
        'best_val_macro_f1': best.get('best_val_macro_f1'),
        'ranking_path': str(rank_csv),
    }, indent=2, ensure_ascii=False), encoding='utf-8')

    print('[OK] Baseline metrics summary:', all_csv)
    print('[OK] Baseline ranking table:', rank_csv)
    print('[OK] Best-model result:', best_json)
    print()
    print(f'===== Baseline Ranking (by {PRIMARY_METRIC}) =====')
    display(ranking[['model_key', 'test_acc', 'test_macro_f1', 'best_val_macro_f1']])
    print('[BEST] {} | test_acc={:.6f}, test_macro_f1={:.6f}'.format(best['model_key'], float(best['test_acc']), float(best['test_macro_f1'])))